In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    DecisionTreeClassifier,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml.evaluation import (
    MulticlassClassificationEvaluator
)

import pandas as pd
import time

In [3]:
spark = SparkSession.builder \
    .appName("NYC Taxi Task3") \
    .getOrCreate()

In [4]:
df = spark.read.parquet(
    "yellow_tripdata_2024-01.parquet",
    "yellow_tripdata_2024-02.parquet",
    "yellow_tripdata_2024-03.parquet",
    "yellow_tripdata_2024-04.parquet"
)

Use 10% Sample

In [5]:
df = df.sample(
    fraction=0.10,
    seed=42
)

print(df.count())

1306203


Missing Values

In [6]:
df = df.fillna({
    "passenger_count":1,
    "RatecodeID":1,
    "congestion_surcharge":0,
    "Airport_fee":0
})

df = df.fillna({
    "store_and_fwd_flag":"N"
})

Feature Engineering

In [7]:
df = df.withColumn(
    "trip_duration",
    (
        unix_timestamp("tpep_dropoff_datetime")
        -
        unix_timestamp("tpep_pickup_datetime")
    ) / 60
)

df = df.withColumn(
    "pickup_hour",
    hour("tpep_pickup_datetime")
)

df = df.withColumn(
    "pickup_day",
    dayofweek("tpep_pickup_datetime")
)

df = df.withColumn(
    "is_weekend",
    when(
        col("pickup_day").isin([1,7]),
        1
    ).otherwise(0)
)

df = df.withColumn(
    "high_tip",
    when(
        col("tip_amount") >= 5,
        1
    ).otherwise(0)
)

Remove Invalid Records

In [8]:
df = df.filter(col("trip_distance") > 0)

df = df.filter(col("fare_amount") > 0)

df = df.filter(col("trip_duration") > 0)

Encoding

In [9]:
indexer = StringIndexer(
    inputCol="payment_type",
    outputCol="payment_type_index"
)

Feature Vector

In [10]:
feature_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "PULocationID",
    "DOLocationID",
    "payment_type_index",
    "pickup_hour",
    "pickup_day",
    "is_weekend",
    "trip_duration"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="assembled_features"
)

scaler = StandardScaler(
    inputCol="assembled_features",
    outputCol="features"
)

Pipeline

In [11]:
pipeline = Pipeline(
    stages=[
        indexer,
        assembler,
        scaler
    ]
)

pipeline_model = pipeline.fit(df)

final_df = pipeline_model.transform(df)

Train/Test Split

In [12]:
train_df, test_df = final_df.randomSplit(
    [0.8,0.2],
    seed=42
)

Evaluation Function

In [13]:
def evaluate(predictions):

    accuracy = MulticlassClassificationEvaluator(
        labelCol="high_tip",
        predictionCol="prediction",
        metricName="accuracy"
    ).evaluate(predictions)

    precision = MulticlassClassificationEvaluator(
        labelCol="high_tip",
        predictionCol="prediction",
        metricName="weightedPrecision"
    ).evaluate(predictions)

    recall = MulticlassClassificationEvaluator(
        labelCol="high_tip",
        predictionCol="prediction",
        metricName="weightedRecall"
    ).evaluate(predictions)

    f1 = MulticlassClassificationEvaluator(
        labelCol="high_tip",
        predictionCol="prediction",
        metricName="f1"
    ).evaluate(predictions)

    return accuracy, precision, recall, f1

Logistic Regression

In [14]:
start = time.time()

lr = LogisticRegression(
    featuresCol="features",
    labelCol="high_tip"
)

lr_model = lr.fit(train_df)

lr_predictions = lr_model.transform(test_df)

lr_metrics = evaluate(
    lr_predictions
)

lr_time = time.time() - start

Decision Tree

In [15]:
start = time.time()

dt = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="high_tip",
    maxDepth=10
)

dt_model = dt.fit(train_df)

dt_predictions = dt_model.transform(test_df)

dt_metrics = evaluate(
    dt_predictions
)

dt_time = time.time() - start

Random Forest

In [16]:
start = time.time()

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="high_tip",
    numTrees=50,
    maxDepth=10
)

rf_model = rf.fit(train_df)

rf_predictions = rf_model.transform(test_df)

rf_metrics = evaluate(
    rf_predictions
)

rf_time = time.time() - start

GBT

In [17]:
start = time.time()

gbt = GBTClassifier(
    featuresCol="features",
    labelCol="high_tip",
    maxDepth=8,
    maxIter=20
)

gbt_model = gbt.fit(train_df)

gbt_predictions = gbt_model.transform(test_df)

gbt_metrics = evaluate(
    gbt_predictions
)

gbt_time = time.time() - start

Results Table

In [18]:
results = pd.DataFrame(
    [
        ["Logistic Regression",*lr_metrics,lr_time],
        ["Decision Tree",*dt_metrics,dt_time],
        ["Random Forest",*rf_metrics,rf_time],
        ["GBT",*gbt_metrics,gbt_time]
    ],
    columns=[
        "Model",
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "Training_Time"
    ]
)

results

,Model,Accuracy,Precision,Recall,F1,Training_Time
0,Logistic Regression,0.879920,0.872363,0.879920,0.867902,100.366816
1,Decision Tree,0.930708,0.931964,0.930708,0.931255,102.907475
2,Random Forest,0.930534,0.931502,0.930534,0.930967,265.036237
3,GBT,0.931351,0.932274,0.931351,0.931765,195.863713


In [19]:
results.to_csv(
    "model_metrics.csv",
    index=False
)

In [20]:
rf_model.write().overwrite().save(
    "best_rf_model"
)